# Attention Playground — coding workshop (advanced)

**Goes with the Session 1 lesson, Step 4 · Attention and Step 6 · Transformers.** Expected
time: ~20 minutes, individual. No GPU needed: everything here runs on the CPU in a few
minutes.

In the lesson you computed attention by hand (0.79 of the weight on the key after *tía*),
and the interactive pages showed heads of a real model, with weights computed in advance.
Here you work with that real model yourself, `Helsinki-NLP/opus-mt-gl-en`:

- **Part A:** find what each attention head does, automatically, on a sentence you choose.
- **Part B:** measure what attention costs as the text gets longer, and find out where the
  compute of a transformer actually goes.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried, or run the
   cell under it: that puts the solution into a new code cell for you, ready to run.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

## Part A — Find the heads

### Load the model

The dedicated Galician→English model from Notebook 1. Its encoder has 6 layers with 8
heads each: 48 self-attention heads.

In [ ]:
!pip install -q transformers sentencepiece codecarbon

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL_NAME = "Helsinki-NLP/opus-mt-gl-en"
tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL_NAME, attn_implementation="eager")
model.eval()  # inference only: no training, no gradients

cfg = model.config
print(f"layers: {cfg.encoder_layers}, heads per layer: {cfg.encoder_attention_heads}, "
      f"dimensions: {cfg.d_model}, feed-forward dimensions: {cfg.encoder_ffn_dim}, "
      f"longest input: {cfg.max_position_embeddings} tokens")

### Every head, on one sentence

`SENTENCE` is our sentence from the lesson. Run the notebook once as it is, then come back
and put in a Galician sentence of your own.

The encoder returns one attention matrix per head: `A[layer, head]` has one row per token
and one column per token. Row *i* holds the weights token *i* gives to every token, and
it adds up to 1.

In [ ]:
SENTENCE = ("Unha tía miña, que todo o solucionaba con ditos e refráns, "
            "dicía algo así como \"un é escravo do que di e dono do que cala\".")

inputs = tokenizer(SENTENCE, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
with torch.no_grad():
    enc_out = model.get_encoder()(**inputs, output_attentions=True)

A = torch.stack(enc_out.attentions)[:, 0].numpy()
print(len(tokens), "tokens:", tokens)
print("A has shape", A.shape, "(layers, heads, tokens, tokens)")

### Score each head

A *previous-word head* puts its weight on the token just before: in row *i*, on column
*i* − 1. So its score is the average of A[*i*, *i* − 1] over all rows that have a previous
token. A *next-word head* uses A[*i*, *i* + 1], and a *self head* A[*i*, *i*]. One
function covers all three, with an `offset` of −1, +1 or 0.

### ✏️ Exercise 1 — Score each head

Complete `pattern_scores`: for every head, the average over all rows *i* of the
weight on column *i* + `offset`, using only rows where that column exists.

<details><summary><b>Hints</b> (click to open)</summary>

- Row i has a partner when `0 <= i + offset < n`.
- The partner column of row i is `i + offset`.

</details>

In [ ]:
def pattern_scores(A, offset):
    """Average weight on the token `offset` positions away, for every head: shape (layers, heads)."""
    n = A.shape[-1]
    # TODO: the rows i whose partner i + offset exists, and the partner columns
    rows = ___
    cols = ___
    return A[:, :, rows, cols].mean(axis=-1)


patterns = {"previous token": -1, "same token": 0, "next token": +1}
scores = {name: pattern_scores(A, off) for name, off in patterns.items()}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (name, s) in zip(axes, scores.items()):
    im = ax.imshow(s, vmin=0, vmax=1, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
    ax.set_xticks(range(s.shape[1]), range(1, s.shape[1] + 1))
    ax.set_yticks(range(s.shape[0]), range(1, s.shape[0] + 1))
fig.colorbar(im, ax=axes, label="average weight")
plt.show()

for name, s in scores.items():
    layer, head = np.unravel_index(s.argmax(), s.shape)
    print(f"best {name} head: layer {layer + 1}, head {head + 1}, score {s.max():.2f}")

In [ ]:
# Check for Exercise 1. Run this cell as is.
def test_shape():
    """one score per head"""
    assert pattern_scores(A, -1).shape == A.shape[:2]

def test_identity():
    """a head that only looks at itself"""
    I = np.eye(5)[None, None]
    assert pattern_scores(I, 0)[0, 0] == 1.0 and pattern_scores(I, -1)[0, 0] == 0.0

def test_shift():
    """a perfect previous-token head"""
    P = np.zeros((1, 1, 4, 4))
    P[0, 0, 0, 0] = 1
    for i in range(1, 4):
        P[0, 0, i, i - 1] = 1
    assert pattern_scores(P, -1)[0, 0] == 1.0, "rows 1, 2, 3 all put weight 1 on i - 1"
    assert abs(pattern_scores(P, +1)[0, 0]) < 1e-12

def test_real_heads():
    """the lesson's heads on the real model"""
    assert np.unravel_index(scores["previous token"].argmax(), (6, 8)) == (5, 6), "layer 6, head 7"
    assert np.unravel_index(scores["next token"].argmax(), (6, 8)) == (2, 3), "layer 3, head 4"

_failed = 0
for _test in [test_shape, test_identity, test_shift, test_real_heads]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def pattern_scores(A, offset):
    """Average weight on the token `offset` positions away, for every head: shape (layers, heads)."""
    n = A.shape[-1]
    rows = [i for i in range(n) if 0 <= i + offset < n]
    cols = [i + offset for i in rows]
    return A[:, :, rows, cols].mean(axis=-1)


patterns = {"previous token": -1, "same token": 0, "next token": +1}
scores = {name: pattern_scores(A, off) for name, off in patterns.items()}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, (name, s) in zip(axes, scores.items()):
    im = ax.imshow(s, vmin=0, vmax=1, cmap="Blues")
    ax.set_title(name)
    ax.set_xlabel("head")
    ax.set_ylabel("layer")
    ax.set_xticks(range(s.shape[1]), range(1, s.shape[1] + 1))
    ax.set_yticks(range(s.shape[0]), range(1, s.shape[0] + 1))
fig.colorbar(im, ax=axes, label="average weight")
plt.show()

for name, s in scores.items():
    layer, head = np.unravel_index(s.argmax(), s.shape)
    print(f"best {name} head: layer {layer + 1}, head {head + 1}, score {s.max():.2f}")
```

</details>

In [ ]:
# Stuck on Exercise 1? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 1" above; your own cell stays as it is.)
get_ipython().set_next_input("def pattern_scores(A, offset):\n    \"\"\"Average weight on the token `offset` positions away, for every head: shape (layers, heads).\"\"\"\n    n = A.shape[-1]\n    rows = [i for i in range(n) if 0 <= i + offset < n]\n    cols = [i + offset for i in rows]\n    return A[:, :, rows, cols].mean(axis=-1)\n\n\npatterns = {\"previous token\": -1, \"same token\": 0, \"next token\": +1}\nscores = {name: pattern_scores(A, off) for name, off in patterns.items()}\n\nfig, axes = plt.subplots(1, 3, figsize=(13, 3.6))\nfor ax, (name, s) in zip(axes, scores.items()):\n    im = ax.imshow(s, vmin=0, vmax=1, cmap=\"Blues\")\n    ax.set_title(name)\n    ax.set_xlabel(\"head\")\n    ax.set_ylabel(\"layer\")\n    ax.set_xticks(range(s.shape[1]), range(1, s.shape[1] + 1))\n    ax.set_yticks(range(s.shape[0]), range(1, s.shape[0] + 1))\nfig.colorbar(im, ax=axes, label=\"average weight\")\nplt.show()\n\nfor name, s in scores.items():\n    layer, head = np.unravel_index(s.argmax(), s.shape)\n    print(f\"best {name} head: layer {layer + 1}, head {head + 1}, score {s.max():.2f}\")")


### Look inside the best previous-token head

In [ ]:
s = scores["previous token"]
layer, head = np.unravel_index(s.argmax(), s.shape)
W = A[layer, head]

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(W, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tokens)), tokens, rotation=90, fontsize=8)
ax.set_yticks(range(len(tokens)), tokens, fontsize=8)
ax.set_xlabel("looks at")
ax.set_ylabel("token")
ax.set_title(f"encoder layer {layer + 1}, head {head + 1}")
plt.show()

i = tokens.index("▁miña") if "▁miña" in tokens else 1
j = int(W[i].argmax())
print(f"{tokens[i]} looks most at {tokens[j]}, with weight {W[i, j]:.2f}")

### ✏️ Exercise 2 — Recompute a head from the model's own weights (advanced)

`A` came out of the model. Recompute layer 1 yourself, from its weights: the
input of layer 1 is `enc_out.hidden_states[0]`; its query and key projections are
`layer.q_proj` and `layer.k_proj`; the model multiplies the queries by `layer.scaling`
(1/√64). Split the 512 dimensions into 8 heads of 64, then softmax(Q Kᵀ) per head.

<details><summary><b>Hints</b> (click to open)</summary>

- `layer.q_proj(h)` has shape (tokens, 512); `.view(n_tok, n_heads, 64)` splits it into heads.
- `.transpose(0, 1)` moves the head dimension first: (heads, tokens, 64).
- Multiply the queries by `layer.scaling` before the dot products.

</details>

In [ ]:
with torch.no_grad():
    enc_out = model.get_encoder()(**inputs, output_attentions=True, output_hidden_states=True)
    layer = model.get_encoder().layers[0].self_attn
    h = enc_out.hidden_states[0][0]                     # (tokens, 512)
    n_tok, n_heads = h.shape[0], cfg.encoder_attention_heads
    # TODO: queries (scaled) and keys, shaped (heads, tokens, 64)
    Q = ___
    K = ___
    my_A = torch.softmax(Q @ K.transpose(1, 2), dim=-1).numpy()

print("largest difference from the model's weights:", np.abs(my_A - A[0]).max())

In [ ]:
# Check for Exercise 2. Run this cell as is.
assert my_A.shape == A[0].shape
assert np.abs(my_A - A[0]).max() < 1e-4, "your weights should match the model's"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
with torch.no_grad():
    enc_out = model.get_encoder()(**inputs, output_attentions=True, output_hidden_states=True)
    layer = model.get_encoder().layers[0].self_attn
    h = enc_out.hidden_states[0][0]
    n_tok, n_heads = h.shape[0], cfg.encoder_attention_heads
    Q = (layer.q_proj(h) * layer.scaling).view(n_tok, n_heads, -1).transpose(0, 1)
    K = layer.k_proj(h).view(n_tok, n_heads, -1).transpose(0, 1)
    my_A = torch.softmax(Q @ K.transpose(1, 2), dim=-1).numpy()

print("largest difference from the model's weights:", np.abs(my_A - A[0]).max())
```

</details>

In [ ]:
# Stuck on Exercise 2? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 2" above; your own cell stays as it is.)
get_ipython().set_next_input("with torch.no_grad():\n    enc_out = model.get_encoder()(**inputs, output_attentions=True, output_hidden_states=True)\n    layer = model.get_encoder().layers[0].self_attn\n    h = enc_out.hidden_states[0][0]\n    n_tok, n_heads = h.shape[0], cfg.encoder_attention_heads\n    Q = (layer.q_proj(h) * layer.scaling).view(n_tok, n_heads, -1).transpose(0, 1)\n    K = layer.k_proj(h).view(n_tok, n_heads, -1).transpose(0, 1)\n    my_A = torch.softmax(Q @ K.transpose(1, 2), dim=-1).numpy()\n\nprint(\"largest difference from the model's weights:\", np.abs(my_A - A[0]).max())")


**Reflection:** Which heads score highest for each pattern? Are they the heads the lesson's
“Self-attention heads” page shows? Now put a Galician sentence of your own into `SENTENCE` and
run Part A again. Do the same heads win? What does that tell you about whether a head's
job belongs to the head or to the sentence?

### Cross-attention: where each English word looks

The decoder writes English, and its cross-attention looks back at the Galician tokens. For
each English token, the table compares the average over all 48 heads with the single head
that looks hardest at one source token.

In [ ]:
with torch.no_grad():
    generated = model.generate(**inputs, max_new_tokens=60)
    dec_out = model(**inputs, decoder_input_ids=generated, output_attentions=True)
print("Translation:", tokenizer.decode(generated[0], skip_special_tokens=True))

C = torch.stack(dec_out.cross_attentions)[:, 0].numpy()
out_tokens = tokenizer.convert_ids_to_tokens(generated[0])[1:]
C = C[:, :, :len(out_tokens), :]

print(f"\n{'English':>12} | {'average of 48 heads':>24} | best single head")
for t, word in enumerate(out_tokens[:12]):
    avg = C[:, :, t].mean(axis=(0, 1))
    j = int(avg.argmax())
    peak = C[:, :, t].max(axis=-1)          # each head's largest weight: (layers, heads)
    layer, head = np.unravel_index(peak.argmax(), peak.shape)
    jb = int(C[layer, head, t].argmax())
    print(f"{word:>12} | {tokens[j]:>14} {avg[j]:.2f} | layer {layer + 1}, head {head + 1}: "
          f"{tokens[jb]} {C[layer, head, t, jb]:.2f}")

t_aunt = next(t for t, w in enumerate(out_tokens) if w.lower() == "▁aunt")
jl = int(C[5, 1, t_aunt].argmax())
print(f"\nlayer 6, head 2, writing 'aunt': {tokens[jl]} {C[5, 1, t_aunt, jl]:.2f}")

**Reflection:** Compare the average column with the best-head column. Which is sharper? Do the
best heads always pick the source word you would align by hand? Why does an average over
48 heads look blurrier than any single head?

## Part B — What does attention cost?

### Counting comparisons

Each head scores every token against every token: n × n scores for n tokens. The encoder
has 48 heads, and stores each score as a 4-byte number.

In [ ]:
n_heads_total = cfg.encoder_layers * cfg.encoder_attention_heads

print(f"{'tokens n':>9} | {'scores per head':>16} | {'all 48 heads':>13} | {'memory, MB':>10}")
for n in [32, 64, 128, 256, 512]:
    per_head = n * n
    total = per_head * n_heads_total
    print(f"{n:>9} | {per_head:>16,} | {total:>13,} | {total * 4 / 1e6:>10.1f}")

Doubling the tokens multiplies the scores by 4: quadratic growth, as in the lesson. Does
the *time* of the whole encoder grow the same way? Measure it.

### Measuring the whole encoder

The input is our sentence repeated until it has n tokens (512 is the longest input this
model accepts). Each length runs several times, and CodeCarbon measures the energy.

In [ ]:
import time
from codecarbon import EmissionsTracker


def get_tracker_energy_kwh(tracker):
    """Total energy of a stopped tracker, in kWh (the attribute moved between CodeCarbon versions)."""
    for attr_path in ("final_emissions_data.energy_consumed", "_total_energy.kWh"):
        obj = tracker
        try:
            for part in attr_path.split("."):
                obj = getattr(obj, part)
            return float(obj)
        except Exception:
            continue
    return None


base_ids = tokenizer(SENTENCE)["input_ids"][:-1]   # without the final </s>
LENGTHS = [32, 64, 128, 256, 512]
REPEATS = 20

encoder = model.get_encoder()
results = []
for n in LENGTHS:
    ids = (base_ids * (n // len(base_ids) + 1))[:n - 1] + [tokenizer.eos_token_id]
    x = torch.tensor([ids])
    with torch.no_grad():
        encoder(input_ids=x)                       # warm-up run, not measured
        tracker = EmissionsTracker(measure_power_secs=1, save_to_file=False, log_level="error")
        tracker.start()
        t0 = time.perf_counter()
        for _ in range(REPEATS):
            encoder(input_ids=x)
        seconds = (time.perf_counter() - t0) / REPEATS
        tracker.stop()
    kwh = get_tracker_energy_kwh(tracker)
    joules = kwh * 3.6e6 / REPEATS if kwh is not None else float("nan")
    results.append((n, seconds, joules))
    print(f"n = {n:>3}: {seconds * 1000:7.1f} ms per pass, {joules:.3f} J per pass")

If the time grew like n², each doubling of n would multiply it by 4. If it grew like n,
by 2.

### ✏️ Exercise 3 — How much does each doubling cost?

`secs[k]` is the time for `ns[k]` tokens, and each length is twice the one before.
Compute `ratios`: time(2n) / time(n), for each doubling.

<details><summary><b>Hints</b> (click to open)</summary>

- `secs[1:]` and `secs[:-1]` line up each length with the one before it.

</details>

In [ ]:
ns = np.array([r[0] for r in results])
secs = np.array([r[1] for r in results])

ratios = ___  # TODO: time(2n) / time(n) for each doubling
for n, r in zip(ns[1:], ratios):
    print(f"{n // 2:>3} -> {n:>3} tokens: time x {r:.2f}")

plt.figure(figsize=(6, 4))
plt.plot(ns, secs * 1000, "o-", label="measured")
plt.plot(ns, secs[0] * 1000 * ns / ns[0], "--", label="if linear (x 2 per doubling)")
plt.plot(ns, secs[0] * 1000 * (ns / ns[0]) ** 2, ":", label="if quadratic (x 4 per doubling)")
plt.ylim(0, secs[-1] * 1000 * 1.6)
plt.xlabel("tokens n")
plt.ylabel("ms per encoder pass")
plt.legend()
plt.show()

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert len(ratios) == len(secs) - 1
assert all(abs(ratios[k] - secs[k + 1] / secs[k]) < 1e-12 for k in range(len(ratios)))
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
ns = np.array([r[0] for r in results])
secs = np.array([r[1] for r in results])

ratios = secs[1:] / secs[:-1]
for n, r in zip(ns[1:], ratios):
    print(f"{n // 2:>3} -> {n:>3} tokens: time x {r:.2f}")

plt.figure(figsize=(6, 4))
plt.plot(ns, secs * 1000, "o-", label="measured")
plt.plot(ns, secs[0] * 1000 * ns / ns[0], "--", label="if linear (x 2 per doubling)")
plt.plot(ns, secs[0] * 1000 * (ns / ns[0]) ** 2, ":", label="if quadratic (x 4 per doubling)")
plt.ylim(0, secs[-1] * 1000 * 1.6)
plt.xlabel("tokens n")
plt.ylabel("ms per encoder pass")
plt.legend()
plt.show()
```

</details>

In [ ]:
# Stuck on Exercise 3? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 3" above; your own cell stays as it is.)
get_ipython().set_next_input("ns = np.array([r[0] for r in results])\nsecs = np.array([r[1] for r in results])\n\nratios = secs[1:] / secs[:-1]\nfor n, r in zip(ns[1:], ratios):\n    print(f\"{n // 2:>3} -> {n:>3} tokens: time x {r:.2f}\")\n\nplt.figure(figsize=(6, 4))\nplt.plot(ns, secs * 1000, \"o-\", label=\"measured\")\nplt.plot(ns, secs[0] * 1000 * ns / ns[0], \"--\", label=\"if linear (x 2 per doubling)\")\nplt.plot(ns, secs[0] * 1000 * (ns / ns[0]) ** 2, \":\", label=\"if quadratic (x 4 per doubling)\")\nplt.ylim(0, secs[-1] * 1000 * 1.6)\nplt.xlabel(\"tokens n\")\nplt.ylabel(\"ms per encoder pass\")\nplt.legend()\nplt.show()")


**Reflection:** Is each doubling closer to × 2 or to × 4? The number of attention scores grows
like n², so what else in the encoder could be taking the time?

### Where the compute goes

Count the multiplications and additions (floating-point operations, FLOPs) in one
encoder layer, for n tokens with d dimensions and a feed-forward layer of d_ff dimensions.
Multiplying a vector of d values by a d × d matrix costs about 2·d² FLOPs.

| part of the layer | FLOPs | grows like |
|---|---|---|
| query, key, value and output projections: 4 matrices of d × d | n · 2 · 4d² | n |
| feed-forward: d × d_ff, then d_ff × d | n · 2 · 2·d·d_ff | n |
| attention scores q · k, for every pair of tokens | n² · 2d | n² |
| weighted sum of the values, for every pair | n² · 2d | n² |

The n² part wins only when n · 4d > 8d² + 4·d·d_ff, that is when
n > 2d + d_ff.

### ✏️ Exercise 4 — Count the FLOPs of a layer

Complete `layer_flops` from the table above: per token, the four d × d projections
cost 2 · 4d² FLOPs and the feed-forward layer 2 · 2 · d · d_ff; for every pair of tokens,
the score q · k costs 2d and the weighted sum another 2d.

<details><summary><b>Hints</b> (click to open)</summary>

- Projections: `d * d` per matrix; feed-forward: `d * d_ff` per matrix, two matrices.
- The quadratic part has one term per pair of tokens: `n * n`.

</details>

In [ ]:
def layer_flops(n, d, d_ff):
    """FLOPs of one encoder layer for n tokens: (part that grows like n, part that grows like n²)."""
    # TODO: the part that grows like n, and the part that grows like n²
    linear = ___
    quadratic = ___
    return linear, quadratic


d, d_ff = cfg.d_model, cfg.encoder_ffn_dim
crossover = 2 * d + d_ff
print(f"opus-mt: d = {d}, d_ff = {d_ff}; the n² part is larger only above n = {crossover:,} tokens")
print(f"\n{'tokens n':>9} | {'share of FLOPs in the n² part':>30}")
for n in [32, 128, 512, 2048, 8192, 32768]:
    lin, quad = layer_flops(n, d, d_ff)
    print(f"{n:>9,} | {quad / (lin + quad):>29.0%}")

In [ ]:
# Check for Exercise 4. Run this cell as is.
_lin, _quad = layer_flops(crossover, d, d_ff)
assert _lin == _quad, "at n = 2d + d_ff the two parts are equal"
assert layer_flops(1, 1, 1) == (12, 4)
_l1, _q1 = layer_flops(100, d, d_ff)
_l2, _q2 = layer_flops(200, d, d_ff)
assert _l2 == 2 * _l1 and _q2 == 4 * _q1, "linear part x 2, quadratic part x 4"
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def layer_flops(n, d, d_ff):
    """FLOPs of one encoder layer for n tokens: (part that grows like n, part that grows like n²)."""
    linear = n * (2 * 4 * d * d + 2 * 2 * d * d_ff)
    quadratic = n * n * (2 * d + 2 * d)
    return linear, quadratic


d, d_ff = cfg.d_model, cfg.encoder_ffn_dim
crossover = 2 * d + d_ff
print(f"opus-mt: d = {d}, d_ff = {d_ff}; the n² part is larger only above n = {crossover:,} tokens")
print(f"\n{'tokens n':>9} | {'share of FLOPs in the n² part':>30}")
for n in [32, 128, 512, 2048, 8192, 32768]:
    lin, quad = layer_flops(n, d, d_ff)
    print(f"{n:>9,} | {quad / (lin + quad):>29.0%}")
```

</details>

In [ ]:
# Stuck on Exercise 4? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 4" above; your own cell stays as it is.)
get_ipython().set_next_input("def layer_flops(n, d, d_ff):\n    \"\"\"FLOPs of one encoder layer for n tokens: (part that grows like n, part that grows like n²).\"\"\"\n    linear = n * (2 * 4 * d * d + 2 * 2 * d * d_ff)\n    quadratic = n * n * (2 * d + 2 * d)\n    return linear, quadratic\n\n\nd, d_ff = cfg.d_model, cfg.encoder_ffn_dim\ncrossover = 2 * d + d_ff\nprint(f\"opus-mt: d = {d}, d_ff = {d_ff}; the n² part is larger only above n = {crossover:,} tokens\")\nprint(f\"\\n{'tokens n':>9} | {'share of FLOPs in the n² part':>30}\")\nfor n in [32, 128, 512, 2048, 8192, 32768]:\n    lin, quad = layer_flops(n, d, d_ff)\n    print(f\"{n:>9,} | {quad / (lin + quad):>29.0%}\")")


### Attention alone

Now time just the attention step, softmax(QKᵀ/√d)·V for one head with 64 dimensions,
far past this model's 512-token limit. Nothing else runs, so the n² part has nothing to
hide behind.

### ✏️ Exercise 5 — Attention, written out

Complete `attention`: scores Q Kᵀ / √d, then a softmax over each row, then the
weighted sum of the values.

<details><summary><b>Hints</b> (click to open)</summary>

- `Q @ K.T` is the matrix of all query-key dot products.
- `weights.sum(axis=1, keepdims=True)` gives one sum per row; `weights @ V` is the weighted sum.

</details>

In [ ]:
def attention(Q, K, V):
    """Scaled dot-product attention for one head: softmax(Q Kᵀ / sqrt(d)) V."""
    # TODO: the scaled scores, one row per query
    scores = ___
    scores -= scores.max(axis=1, keepdims=True)   # for numerical stability
    weights = np.exp(scores)
    # TODO: normalize each row to sum 1, then the weighted sum of the values
    ___


rng = np.random.default_rng(0)
att_lengths = [512, 1024, 2048, 4096, 8192]
att_secs = []
for n in att_lengths:
    Q, K, V = (rng.standard_normal((n, 64), dtype=np.float32) for _ in range(3))
    attention(Q, K, V)                             # warm-up
    t0 = time.perf_counter()
    for _ in range(3):
        attention(Q, K, V)
    att_secs.append((time.perf_counter() - t0) / 3)
    print(f"n = {n:>5}: {att_secs[-1] * 1000:8.1f} ms")

for n, r in zip(att_lengths[1:], np.array(att_secs[1:]) / np.array(att_secs[:-1])):
    print(f"{n // 2:>5} -> {n:>5} tokens: time x {r:.2f}")

In [ ]:
# Check for Exercise 5. Run this cell as is.
def test_against_torch():
    """matches PyTorch's scaled_dot_product_attention"""
    _r = np.random.default_rng(1)
    Q_, K_, V_ = (_r.standard_normal((7, 16)).astype(np.float32) for _ in range(3))
    ref = torch.nn.functional.scaled_dot_product_attention(*(torch.tensor(x)[None] for x in (Q_, K_, V_)))[0].numpy()
    assert np.allclose(attention(Q_, K_, V_), ref, atol=1e-5)

def test_quadratic():
    """attention alone grows faster than linearly"""
    assert att_secs[-1] / att_secs[0] > 16 * 1.5, "16 times the tokens should take far more than 16 times as long"

_failed = 0
for _test in [test_against_torch, test_quadratic]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
def attention(Q, K, V):
    """Scaled dot-product attention for one head: softmax(Q Kᵀ / sqrt(d)) V."""
    scores = Q @ K.T / np.sqrt(Q.shape[1])
    scores -= scores.max(axis=1, keepdims=True)   # for numerical stability
    weights = np.exp(scores)
    weights /= weights.sum(axis=1, keepdims=True)
    return weights @ V


rng = np.random.default_rng(0)
att_lengths = [512, 1024, 2048, 4096, 8192]
att_secs = []
for n in att_lengths:
    Q, K, V = (rng.standard_normal((n, 64), dtype=np.float32) for _ in range(3))
    attention(Q, K, V)                             # warm-up
    t0 = time.perf_counter()
    for _ in range(3):
        attention(Q, K, V)
    att_secs.append((time.perf_counter() - t0) / 3)
    print(f"n = {n:>5}: {att_secs[-1] * 1000:8.1f} ms")

for n, r in zip(att_lengths[1:], np.array(att_secs[1:]) / np.array(att_secs[:-1])):
    print(f"{n // 2:>5} -> {n:>5} tokens: time x {r:.2f}")
```

</details>

In [ ]:
# Stuck on Exercise 5? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 5" above; your own cell stays as it is.)
get_ipython().set_next_input("def attention(Q, K, V):\n    \"\"\"Scaled dot-product attention for one head: softmax(Q Kᵀ / sqrt(d)) V.\"\"\"\n    scores = Q @ K.T / np.sqrt(Q.shape[1])\n    scores -= scores.max(axis=1, keepdims=True)   # for numerical stability\n    weights = np.exp(scores)\n    weights /= weights.sum(axis=1, keepdims=True)\n    return weights @ V\n\n\nrng = np.random.default_rng(0)\natt_lengths = [512, 1024, 2048, 4096, 8192]\natt_secs = []\nfor n in att_lengths:\n    Q, K, V = (rng.standard_normal((n, 64), dtype=np.float32) for _ in range(3))\n    attention(Q, K, V)                             # warm-up\n    t0 = time.perf_counter()\n    for _ in range(3):\n        attention(Q, K, V)\n    att_secs.append((time.perf_counter() - t0) / 3)\n    print(f\"n = {n:>5}: {att_secs[-1] * 1000:8.1f} ms\")\n\nfor n, r in zip(att_lengths[1:], np.array(att_secs[1:]) / np.array(att_secs[:-1])):\n    print(f\"{n // 2:>5} -> {n:>5} tokens: time x {r:.2f}\")")


### ✏️ Exercise 6 — Attention in chunks: the same result in less memory (advanced)

`attention` builds the whole n × n score matrix: for 8,192 tokens that is 67
million numbers. Compute the same output a block of queries at a time, so only a
`chunk` × n block exists at once. (Memory-efficient attention kernels such as
FlashAttention push this idea further.)

<details><summary><b>Hints</b> (click to open)</summary>

- Each query row only needs its own scores: rows `start:start + chunk` of Q, with all of K and V.
- Reuse your `attention` function on that block of queries.

</details>

In [ ]:
def chunked_attention(Q, K, V, chunk=512):
    out = np.empty((Q.shape[0], V.shape[1]), dtype=np.float32)
    for start in range(0, Q.shape[0], chunk):
        # TODO: the output rows for queries start … start + chunk
        out[start:start + chunk] = ___
    return out


Q, K, V = (rng.standard_normal((8192, 64), dtype=np.float32) for _ in range(3))
full = attention(Q, K, V)
part = chunked_attention(Q, K, V)
print("largest difference:", np.abs(full - part).max())
print(f"largest score block: full {8192 * 8192 * 4 / 1e6:.0f} MB, chunked {512 * 8192 * 4 / 1e6:.0f} MB")

In [ ]:
# Check for Exercise 6. Run this cell as is.
assert np.allclose(full, part, atol=1e-5)
_q = rng.standard_normal((1000, 8), dtype=np.float32)
assert np.allclose(chunked_attention(_q, _q, _q, chunk=300), attention(_q, _q, _q), atol=1e-5), \
    "the last chunk may be shorter than `chunk`"
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
def chunked_attention(Q, K, V, chunk=512):
    out = np.empty((Q.shape[0], V.shape[1]), dtype=np.float32)
    for start in range(0, Q.shape[0], chunk):
        out[start:start + chunk] = attention(Q[start:start + chunk], K, V)
    return out


Q, K, V = (rng.standard_normal((8192, 64), dtype=np.float32) for _ in range(3))
full = attention(Q, K, V)
part = chunked_attention(Q, K, V)
print("largest difference:", np.abs(full - part).max())
print(f"largest score block: full {8192 * 8192 * 4 / 1e6:.0f} MB, chunked {512 * 8192 * 4 / 1e6:.0f} MB")
```

</details>

In [ ]:
# Stuck on Exercise 6? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 6" above; your own cell stays as it is.)
get_ipython().set_next_input("def chunked_attention(Q, K, V, chunk=512):\n    out = np.empty((Q.shape[0], V.shape[1]), dtype=np.float32)\n    for start in range(0, Q.shape[0], chunk):\n        out[start:start + chunk] = attention(Q[start:start + chunk], K, V)\n    return out\n\n\nQ, K, V = (rng.standard_normal((8192, 64), dtype=np.float32) for _ in range(3))\nfull = attention(Q, K, V)\npart = chunked_attention(Q, K, V)\nprint(\"largest difference:\", np.abs(full - part).max())\nprint(f\"largest score block: full {8192 * 8192 * 4 / 1e6:.0f} MB, chunked {512 * 8192 * 4 / 1e6:.0f} MB\")")


**Reflection:** Does attention alone grow like × 2 or × 4 per doubling? Above how many tokens
would the n² part dominate opus-mt? Look up the d and d_ff of a larger model (for example
in its `config.json` on Hugging Face) and compute its crossover too. For the inputs in this
course, does cost grow quadratically or roughly linearly with the number of tokens?

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.